# Multilingual Social Media Analytics

Explore a synthetic smartphone-launch dataset in English, Hindi and Hinglish. Run the cells from top to bottom.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from preprocessing import normalize_text, tokenize, detect_language
from sentiment import lexicon_sentiment, theme_tags
SEED = 42


In [ ]:
posts = pd.read_csv(ROOT / "data/social_posts_synthetic.csv")
evaluation = pd.read_csv(ROOT / "data/evaluation_sample_200.csv")
print(posts.shape)
posts.head()


In [ ]:
posts["clean_text"] = posts["text"].map(normalize_text)
posts["tokens"] = posts["text"].map(tokenize)
posts["language_pred"] = posts["text"].map(detect_language)
posts["sentiment_pred"] = posts["text"].map(lexicon_sentiment)
posts["themes"] = posts["text"].map(lambda x: ", ".join(theme_tags(x)))
posts[["text", "clean_text", "language_gold", "language_pred", "sentiment_gold", "sentiment_pred"]].head(10)


In [ ]:
evaluation["language_pred"] = evaluation["text"].map(detect_language)
evaluation["sentiment_pred"] = evaluation["text"].map(lexicon_sentiment)
print("Language accuracy:", accuracy_score(evaluation.language_gold, evaluation.language_pred))
print("Sentiment macro F1:", f1_score(evaluation.sentiment_gold, evaluation.sentiment_pred, average="macro"))
print(classification_report(evaluation.sentiment_gold, evaluation.sentiment_pred, zero_division=0))


## Important evaluation note

The supplied labels are synthetic template labels, not human annotations. Review and correct the 200 examples before using the metrics as formal results. The baseline is intentionally simple and may confuse Romanised Hindi with English.

In [ ]:
language_summary = posts.groupby("language_gold").agg(posts=("post_id", "count")).reset_index()
sentiment_summary = pd.crosstab(posts["language_gold"], posts["sentiment_gold"], normalize="index").round(3)
display(language_summary)
display(sentiment_summary)


In [ ]:
theme_records = []
for _, row in posts.iterrows():
    for theme in theme_tags(row["text"]):
        theme_records.append({"language": row["language_gold"], "sentiment": row["sentiment_gold"], "theme": theme})
theme_summary = pd.DataFrame(theme_records).groupby(["language","sentiment","theme"]).size().rename("posts").reset_index()
theme_summary.sort_values("posts", ascending=False).head(20)


## Transformer comparison (optional)

For a proper second model, install `transformers` and `torch`, load a multilingual sentiment checkpoint such as a multilingual XLM-R sentiment model, and predict on the exact same manually reviewed evaluation split. Record the model name/version and do not compare scores from different test sets. Model downloads may be large.